<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_70_llm_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📏 **Module 70 — LLM Evals (lm-eval-harness · Promptfoo · Langfuse · Phoenix)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 70 — LLM Evals

> **"Did this prompt change make the product better or worse?"** is the question every team gets wrong every week — because they ship by **vibes** instead of **metrics**. Eval is what turns a chatbot from a demo into a product. This module is the **complete eval stack**: academic benchmarks via **lm-eval-harness**, prompt regression via **Promptfoo**, production tracing + LLM-as-judge via **Langfuse**, exploration + curation via **Arize Phoenix**, plus the unwritten rules every senior LLM engineer follows.

### What you'll cover
1. The five eval flavours — when to use which
2. Build your **golden set** (the single most important asset)
3. **Reference metrics** (when they actually work) — exact match, ROUGE, BLEU
4. **LLM-as-judge** — the modern default (and how to keep it honest)
5. **`lm-eval-harness`** — academic benchmarks (MMLU, GSM8K, HumanEval, IFEval, GPQA)
6. **Promptfoo** — prompt regression in YAML + CI
7. **Langfuse + Phoenix** — production evals on real traffic
8. **Online evals** — A/B testing LLM apps for real
9. Eval **anti-patterns** — contamination, judge bias, saturation, Goodhart
10. The eval **playbook** every shipping LLM team should follow


## 1 · The five flavours of LLM eval

| Flavour | What it measures | When |
|---|---|---|
| **Reference-metric** (BLEU/ROUGE/exact) | similarity to a gold answer | classification, extraction, deterministic outputs |
| **Programmatic checks** | "does this output parse as JSON? does this code run? does this regex match?" | structured outputs, code, tool calls |
| **LLM-as-judge** | a stronger model scores your model's output | open-ended generation (most chat use cases) |
| **Human eval** | humans score on a rubric | the most accurate; expensive; reserve for high-stakes decisions |
| **Online metrics** | live KPIs (CTR, deflection, completion rate, retention) | what the business actually cares about |

The trick is the **layered eval pyramid**: fast cheap checks (programmatic) → medium-cost regression (LLM judge) → expensive infrequent validation (human + online). Run cheap evals on every PR; expensive ones on every release.

## 2 · The golden set — your one critical asset

Before any tool, before any framework, you need **the golden set**: a curated `(input, expected output, metadata)` dataset that represents your real traffic. Without it, every eval lies.

### How to build one
1. **Sample 100-300 real production traces.** (M51 OTel makes this trivial.)
2. **Stratify by intent / persona** — don't oversample easy queries.
3. **Label by hand** — yes, by hand. With the eventual rubric you'll automate.
4. **Tag each example** with category, difficulty, language, contains-PII flag.
5. **Version it** in git. Re-curate quarterly.

### What "golden" should cover
- ✅ The common case (~50% of traffic — what most users do)
- ✅ Known failure modes (every bug fix → add the row to the golden set)
- ✅ Adversarial / jailbreak attempts
- ✅ Multi-turn dialogues (not just single-shot Q&A)
- ✅ "Should refuse" cases — the negative class

## 3 · Reference metrics — when they work, when they don't

In [ ]:
!pip -q install "datasets" "evaluate" rouge_score nltk

In [ ]:
# Reference metrics work when there's a single right answer.
# They fall apart on open-ended generation.

import evaluate
exact   = evaluate.load("exact_match")
rouge   = evaluate.load("rouge")
bleu    = evaluate.load("bleu")

preds = [
    "Paris",
    "The capital of France is Paris.",
    "It is Paris, the City of Light.",
]
refs  = ["Paris"] * 3

print("exact :", exact.compute(predictions=preds, references=refs))
print("rouge :", rouge.compute(predictions=preds, references=refs))
print("bleu  :", bleu.compute(predictions=preds, references=[[r] for r in refs]))

**Read the output.** Three semantically equivalent answers give wildly different scores. Reference metrics:

| Metric | When it works | When it fails |
|---|---|---|
| **Exact match** | classification, structured extraction | any free-form generation |
| **ROUGE-1/2/L** | summarisation against curated references | paraphrase, multiple-valid-answers |
| **BLEU** | machine translation | LLM chat |
| **chrF / METEOR** | translation with character-level robustness | still fails on style variation |

**Rule:** if your output can be paraphrased, **don't use reference metrics**. Use LLM-as-judge instead. If your output has *one* right answer (a digit, a JSON field, a code branch), reference metrics are perfect — and free.

## 4 · LLM-as-judge — the modern default

Most chat outputs can't be reference-matched. Instead, **another LLM grades them**. The standard recipe — single-turn, pair-wise, and rubric — popularised by MT-Bench, AlpacaEval, Arena-Hard.

In [ ]:
# LLM-as-judge prompt template (the one most papers use)
JUDGE_PROMPT = '''You are an impartial judge.
Compare the two responses to the question below.
Pick the one that is more {axis}.

[Question]
{question}

[Response A]
{a}

[Response B]
{b}

Output exactly one of: A, B, TIE.
Reasoning first; final answer on its own line as: VERDICT: <A|B|TIE>'''

def judge_pair(question, a, b, axis="helpful and correct", model_call=None):
    prompt = JUDGE_PROMPT.format(question=question, a=a, b=b, axis=axis)
    out = model_call(prompt)
    # parse the verdict robustly
    verdict_line = next((l for l in out.splitlines() if l.upper().startswith("VERDICT:")), "")
    verdict = verdict_line.upper().split(":")[-1].strip()
    return verdict if verdict in {"A","B","TIE"} else "TIE"

# Use the model_call function from M38/M44/M64 to plug in your judge.

### Five rules to keep your judge honest

1. **Pairwise > absolute scores.** Models are notoriously bad at calibrating "give a 7/10." They're decent at "which of these two is better." Always prefer A-vs-B.
2. **Swap A/B and average.** Judges have a **position bias** — they prefer the first option ~10% more often. Run each pair twice with swapped order and average.
3. **Use a stronger model than the one being judged.** Judging Llama-8B with GPT-4o is fine; judging GPT-4o with Llama-8B is not.
4. **Calibrate on a small human-labelled subset.** If your judge agrees with humans on <80% of a 50-row check set, the judge prompt is broken.
5. **Don't use the same model as the one you're optimising for.** Judging GPT-4 outputs with GPT-4 inflates self-similar style.

> Two model-as-judge specs you should know:
> - **Pairwise** — "which is better?" → win rate
> - **Reference-based** — "score this 0-10 against this gold answer" → mean score
> - **Rubric** — "rate from 1-5 on factuality, helpfulness, harm" → per-axis dashboard

## 5 · lm-eval-harness — academic benchmarks

In [ ]:
!pip -q install "lm-eval[hf]==0.4.3"

**`lm-evaluation-harness`** (EleutherAI) is the canonical academic eval runner. Every model card, every leaderboard, every paper benchmarks against its task implementations. If you publish a model, you publish lm-eval-harness numbers.

In [ ]:
sample_command = '''
# evaluate a HuggingFace model on five common benchmarks
$ lm_eval \
    --model hf \
    --model_args pretrained=meta-llama/Llama-3.2-1B,dtype=bfloat16 \
    --tasks mmlu,hellaswag,arc_challenge,gsm8k,humaneval \
    --batch_size auto \
    --output_path results/llama-3.2-1b

# evaluate via a hosted OpenAI-compatible endpoint (your vLLM from M44)
$ lm_eval \
    --model openai-chat-completions \
    --model_args model=qwen2.5-7b-instruct,base_url=http://localhost:8000/v1,api_key=x \
    --tasks gpqa,ifeval,gsm8k

# Programmatic API
import lm_eval
res = lm_eval.simple_evaluate(
    model="hf",
    model_args="pretrained=gpt2",
    tasks=["arc_easy"],
    limit=20,
)
print(res["results"])
'''
print(sample_command)

### The benchmark cheat sheet

| Benchmark | What it tests | Status in 2026 |
|---|---|---|
| **MMLU** (57 subjects) | broad academic knowledge | nearly saturated — frontier > 89% |
| **HellaSwag** | commonsense completion | saturated |
| **ARC-Challenge** | grade-school science | saturated |
| **WinoGrande** | pronoun resolution | saturated |
| **GSM8K** | grade-school math | saturated by reasoning models |
| **MATH** | competition math | hard; reasoning models needed |
| **HumanEval** | Python from docstring | saturated; replaced by LiveCodeBench |
| **MBPP** | Python from problem statement | saturated |
| **TriviaQA** | open-domain QA | saturated |
| **IFEval** | strict instruction-following (format, length) | still discriminating |
| **GPQA Diamond** | Google-proof PhD QA | discriminating |
| **AIME 2024** | competition math | discriminating; reasoning-model headline |
| **SimpleBench** | trick questions where common-sense wins | discriminating; small models often beat large |
| **LiveBench / LiveCodeBench** | continuously refreshed; harder to contaminate | discriminating |
| **MT-Bench / Arena-Hard / WildBench** | chat quality via LLM-as-judge | use these for chat-tuned models |
| **HELM** | Stanford's holistic suite | runs many of the above |

**Rule of thumb 2026:** for **frontier models** lean on **GPQA Diamond, AIME, SimpleBench, LiveBench, IFEval, Arena-Hard**. The "classical" set (MMLU, HellaSwag, HumanEval) is mostly **contamination + saturation**.

## 6 · Promptfoo — prompt regression in YAML + CI

In [ ]:
!pip -q install --upgrade promptfoo

**Promptfoo** is the most popular prompt-testing tool. It treats prompts like code: you write **YAML test specs** with assertions; CI runs them on every PR; reports show diffs vs the previous baseline.

In [ ]:
promptfoo_yaml = '''
# promptfooconfig.yaml
prompts:
  - "Summarise this in one sentence: {{text}}"
  - "Write a 1-sentence summary of: {{text}}"

providers:
  - id: openai:gpt-4o-mini
  - id: ollama:chat:qwen2.5:0.5b-instruct     # local M38
  - id: openai:gpt-4o                           # the judge

tests:
  - vars:
      text: "The quick brown fox jumps over the lazy dog."
    assert:
      - type: contains
        value: "fox"
      - type: javascript
        value: "output.length < 200"

  - vars:
      text: "Mass produces general relativity's spacetime curvature."
    assert:
      - type: llm-rubric
        value: "The summary preserves the cause-effect relation between mass and curvature."
      - type: factuality
        value: "Mass causes spacetime to curve, which is what we perceive as gravity."

  - vars: { text: "1+1" }
    assert:
      - type: not-contains
        value: "violence"
'''
print(promptfoo_yaml)

**Promptfoo gives you, out of the box.**
- **Side-by-side prompt × provider grid** — try N prompts on M models in one run.
- **Assertion types**: `equals`, `contains`, `regex`, `is-json`, `javascript` (custom JS), `llm-rubric` (judge), `factuality`, `latency`, `cost`.
- **`promptfoo eval` in CI** — fails the build if pass-rate drops below threshold.
- **`promptfoo view`** — local web UI with diffs across runs.
- **Datasets** — point at a CSV / Google Sheet of `(input, expected)` rows.

This is **the** answer to "did my prompt change make things better?" Run it on every PR.

## 7 · Langfuse + Phoenix — production evals on real traffic

Once your app is live, the question becomes: **how well is it doing on production traffic?** Two open-source tools that have eaten this space:

### Langfuse
- **Tracing** — every LLM call, every tool call, captured as nested spans (overlaps with OTel from M51; Langfuse accepts OTLP).
- **Datasets** — promote real production traces into reusable eval datasets.
- **Online evaluators** — define scorers (LLM-as-judge, regex, code execution) that run on every production trace.
- **Prompt management** — versioned prompts pulled from the SDK; A/B between prompt versions.
- **Cost + latency analytics** per model / per user / per feature.

### Arize Phoenix
- Open-source LLM observability + evaluation.
- Built-in evaluators: `RAG hallucination`, `relevance`, `Q&A correctness`, `toxicity`.
- **Cluster visualisation** — UMAP of embeddings; find clusters where your model fails.
- Tight integration with `llama-index` / `langchain` callbacks.

In [ ]:
# Sketch — instrument an LLM call so each request becomes a trace + dataset row
langfuse_sketch = '''
from langfuse.decorators import observe, langfuse_context

@observe()
def rag_chat(user_query: str):
    docs = retriever.search(user_query, k=4)
    answer = llm.complete(make_prompt(user_query, docs))

    # attach metadata for later filtering
    langfuse_context.update_current_observation(
        input=user_query,
        output=answer,
        metadata={"n_docs": len(docs)},
    )
    return answer

# Every call now produces a trace. Promote 200 of them to a dataset and
# run an offline batch eval with an LLM-as-judge scorer.
'''
print(langfuse_sketch)

**The two-loop pattern** every shipping LLM team converges on:

```
   PR     ──► Promptfoo regression on golden set ──► fail-fast CI
   merge  ──► deploy to prod
   prod   ──► Langfuse traces everything
   nightly──► sample traces → judge → metrics
   weekly ──► curate new failures into golden set ──► loop
```

## 8 · Online evals — A/B testing LLM apps

Offline evals say "the new prompt is 4 % better on my golden set." **Online evals** ask "did real users prefer it?" Same A/B-testing machinery as classic web (M28's stats), with LLM-specific gotchas.

### Three signals you can A/B
- **Behavioural** — completion rate, escalation rate, retry rate, session length.
- **Explicit** — thumbs up/down, in-product rating.
- **Implicit** — copy-button clicks, "regenerate" clicks (negative signal), follow-up question rate.

### LLM-specific A/B gotchas
- **Latency confound** — if variant B is 200 ms slower, completion rate drops *whether or not* the output is better. **Match latency** in the test.
- **Cost confound** — variant B uses GPT-4o; A uses Haiku. Different cost ≠ different quality lift. Track *cost-adjusted* lift.
- **Long-tail effect** — a new model may be **worse on common queries but better on edge cases**. Don't average over heterogeneous traffic; segment.
- **Eval contamination** — never reuse a golden-set query in a live A/B as a "validation" call; users will see it.
- **Length bias** — variant that writes longer answers gets more positive feedback regardless of correctness. Strip length effects before scoring.

## 9 · Anti-patterns

| Anti-pattern | Why it bites | Fix |
|---|---|---|
| **Benchmark contamination** | model's training set saw the test questions; scores are inflated | use *fresh* benchmarks (LiveBench, AIME-new, SimpleBench), test on held-out internal data |
| **Goodhart's law** | optimising for the metric warps the model | rotate benchmarks; prioritise online signals |
| **Single-prompt evals** | one prompt template hides robustness | eval across paraphrases of the same intent |
| **Judge bias** | judge prefers verbose answers / its own outputs | swap order, calibrate vs humans, prefer pairwise |
| **No baseline** | "70%" is meaningless without "vs what?" | always show baseline (prior prompt / version / model) |
| **Tiny eval set** | 20 examples ⇒ ±15% noise band | aim for 200-1000 per category, especially for LLM-judge runs |
| **No regression suite** | every bug recurs | every fixed bug becomes a golden-set row |
| **Eval-only-once** | model drifts; vendor patches; you stop measuring | nightly cron on production sample |
| **Reference-metric on chat** | BLEU/ROUGE wrong-shaped for open generation | LLM-as-judge instead |
| **Pass-rate fixation** | binary pass/fail loses signal | track judge **score distributions**, not pass% only |

### Two notable failure modes in 2026
- **Reasoning-model contamination** — many reasoning models trained on solutions of MATH / AIME / GPQA. Headline numbers ≠ real reasoning.
- **Judge collapse** — LLM-as-judge scores cluster near the top of the scale once both candidate models are strong. Switch to pairwise + reference-anchored.

## 10 · The eval playbook every shipping LLM team should follow

Adopt these six practices in order. Most teams stop at step 2 and pay for it later.

1. **Curate a 200-row golden set** stratified by intent. Version in git. Re-curate quarterly.
2. **Promptfoo on every PR** with hard assertions + LLM-rubric. Fail the build under threshold.
3. **`lm-eval-harness` on every release** for the relevant benchmarks (small set, not the whole zoo).
4. **Production tracing in Langfuse / Phoenix.** Every LLM call captured; nightly sampled judge scoring.
5. **Online A/B framework** for changes that exceed a 2 % offline win — half of those won't replicate.
6. **Weekly eval review** — read 20 failing traces; add the patterns to the golden set; loop.

> **The honest take.** Tools matter less than **discipline**: ship nothing that hasn't been eval'd on a static golden set; track everything in prod; review failures every week. Teams that do this dominate teams that ship by vibes — even when the vibe-shippers have a better base model.

## ✅ Recap
- Five flavours: **reference metrics · programmatic · LLM-as-judge · human · online**.
- The **golden set** is your most important asset. Build it before any tool.
- **Reference metrics** for deterministic outputs; **LLM-as-judge** for open generation; **pairwise + swap + calibrate**.
- **`lm-eval-harness`** for academic benchmarks; favour **fresh** ones (GPQA, AIME, SimpleBench, LiveBench, IFEval).
- **Promptfoo** runs prompt regression in CI; **Langfuse / Phoenix** trace + score production.
- Online A/Bs reveal what offline misses. Match latency, segment by intent, watch length bias.
- **Anti-patterns to memorise**: contamination, Goodhart, judge bias, no baseline, tiny eval sets.
- Discipline > tooling. Make eval a weekly habit, not a release-time chore.

Next: **M71 — Triton Inference Server**.
